In [10]:
import urllib.parse
import feedparser

def fetch_arxiv(query, max_results=100):
    base_url = "http://export.arxiv.org/api/query?"

    encoded_query = urllib.parse.quote(query)

    url = f"{base_url}search_query={encoded_query}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending"

    print("DEBUG URL:", url)

    feed = feedparser.parse(url)

    papers = []
    for entry in feed.entries:

        # ✅ safer PDF extraction
        pdf_url = ""
        for link in entry.links:
            if link.type == "application/pdf":
                pdf_url = link.href

        papers.append({
            "id": entry.id,
            "title": entry.title.strip() if hasattr(entry, "title") else "",
            "authors": [author.name for author in entry.authors] if hasattr(entry, "authors") else [],
            "published": entry.published,
            "summary": entry.summary.strip() if hasattr(entry, "summary") else "",
            "pdf_url": pdf_url
        })

    return papers

In [11]:
from datetime import datetime

def filter_last_10_years(papers):
    current_year = datetime.now().year
    filtered = []

    for p in papers:
        try:
            year = int(p.get("published", "")[:4])
            if year >= current_year - 10:
                filtered.append(p)
        except:
            continue  # skip bad records

    return filtered

In [12]:
import requests
import time

def fetch_semantic_scholar(query, limit=100, api_key=None):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": query,
        "limit": limit,
        "fields": "title,authors,year,abstract,citationCount,url"
    }

    headers = {
        "User-Agent": "research-assistant"
    }

    if api_key:
        headers["x-api-key"] = api_key

    for attempt in range(3):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)

            if response.status_code == 200:
                data = response.json()

                papers = []
                for p in data.get("data", []):
                    papers.append({
                        "title": p.get("title"),
                        "authors": [a["name"] for a in p.get("authors", [])],
                        "year": p.get("year"),
                        "abstract": p.get("abstract"),
                        "citation_count": p.get("citationCount"),
                        "url": p.get("url")
                    })

                return papers

            elif response.status_code == 429:
                print("Rate limited... retrying")
                time.sleep(3)

            else:
                print(f"Error: {response.status_code}")
                break

        except requests.exceptions.RequestException as e:
            print("Request failed:", e)
            time.sleep(2)

    return []

In [15]:
from datetime import datetime

if __name__ == "__main__":

    query = '(cat:cs.AI OR cat:cs.LG OR cat:cs.CL) AND (transformer OR "large language model" OR diffusion OR generative)'

    print("Fetching from arXiv...")
    arxiv_papers = fetch_arxiv(query, max_results=2000)

    print(f"Fetched: {len(arxiv_papers)} papers")

    # ✅ Clean + transform first
    cleaned_papers = []
    for p in arxiv_papers:
        try:
            cleaned_papers.append({
                "id": p.get("id"),
                "title": p.get("title", "").strip(),
                "authors": p.get("authors", []),
                "abstract": p.get("summary", "").strip(),
                "year": int(p.get("published", "")[:4]),
                "pdf_url": p.get("pdf_url", "")
            })
        except:
            continue  # skip bad records

    # ✅ Filter AFTER cleaning
    current_year = datetime.now().year
    cleaned_papers = [
        p for p in cleaned_papers
        if p["year"] >= current_year - 10
    ]

    print(f"After filtering (last 10 years): {len(cleaned_papers)}")

    # ✅ Safe print
    if cleaned_papers:
        print("\nSample Paper:")
        print(cleaned_papers[0])
    else:
        print("No papers found")

Fetching from arXiv...
DEBUG URL: http://export.arxiv.org/api/query?search_query=%28cat%3Acs.AI%20OR%20cat%3Acs.LG%20OR%20cat%3Acs.CL%29%20AND%20%28transformer%20OR%20%22large%20language%20model%22%20OR%20diffusion%20OR%20generative%29&start=0&max_results=2000&sortBy=submittedDate&sortOrder=descending
Fetched: 2000 papers
After filtering (last 10 years): 2000

Sample Paper:
{'id': 'http://arxiv.org/abs/2604.22749v1', 'title': 'Representational Harms in LLM-Generated Narratives Against Global Majority Nationalities', 'authors': ['Ilana Nguyen', 'Harini Suresh', 'Thema Monroe-White', 'Evan Shieh'], 'abstract': "Large language models (LLMs) are increasingly used for text generation tasks from everyday use to high-stakes enterprise and government applications, including simulated interviews with asylum seekers. While many works highlight the new potential applications of LLMs, there are risks of LLMs encoding and perpetuating harmful biases about non-dominant communities across the globe. 

In [16]:
import pandas as pd

# Convert to DataFrame
df = pd.DataFrame(cleaned_papers)

# Save to CSV
df.to_csv("research_papers.csv", index=False, encoding="utf-8")

print("✅ Data saved to research_papers.csv")

✅ Data saved to research_papers.csv
